In [ ]:
# %%
"""
Sediment Core Proxy Analysis
Cores 13 & 17 — Lake Wānaka

Usage
-----
1. Edit PROXY_CONFIG to rename axis labels without touching plot code.
2. Edit the EVENT_DEPOSITS dicts to add/remove shaded event intervals.
3. Run all cells top-to-bottom, or run individual cells as needed.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import Patch
from scipy.interpolate import interp1d
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns

# =============================================================================
# CONFIGURATION — edit labels and events here, not in the plot functions
# =============================================================================

# Proxy display names — change these to update ALL plots at once
PROXY_CONFIG = {
    "Mean":     "Grain Size Mean (µm)",
    "CN_Ratio": "C/N Ratio",
    "MagSus":   "Mag. Susceptibility",
    "L_star":   "L* (lightness)",
    "Fe":       "Fe (ppm)",
    "SIO":      "SiO Concentration (ppm)",
}

# Event deposits: list of (min_depth_mm, max_depth_mm) tuples.
# Add or remove tuples to shade different intervals.
EVENT_DEPOSITS_C13 = [
    # (120, 135),   # example event — uncomment or add your own
]

EVENT_DEPOSITS_C17 = [
    # (160, 175),
]

# Colours for each proxy — edit here to change across all plots
PROXY_COLOURS = {
    "Mean":     "steelblue",
    "CN_Ratio": "darkorange",
    "MagSus":   "darkgreen",
    "L_star":   "purple",
    "Fe":       "red",
    "SIO":      "red",
}

# =============================================================================
# DATA LOADING
# =============================================================================

def load_core13(path="Core13.xlsx"):
    df = pd.read_excel(path, header=0, skiprows=[1])
    df.columns = [
        "Sample", "Depth", "DepthMidpoint",
        "ParticleWeight", "CN_Weight", "Epoch",
        "Mean", "Sorting", "Skewness", "Kurtosis",
        "CN_Ratio",
        "MagDepth", "MagSus",
        "L_Depth", "L_star",
        "Fe",
    ]
    df = _clean(df)
    return df


def load_core17(path="Core17.xlsx"):
    df = pd.read_excel(path, header=0, skiprows=[1])
    df.columns = [
        "Sample", "Depth", "D2", "DepthMidpoint",
        "ParticleWeight", "CN_Weight", "Epoch",
        "Mean", "Sorting", "Skewness", "Kurtosis",
        "CN_Ratio",
        "MagDepth", "MagSus",
        "L_Depth", "L_star",
        "Fe",
    ]
    df = _clean(df)
    return df


def _clean(df):
    """Strip non-breaking spaces and coerce numeric columns."""
    df = df.replace(r"^\s+$", np.nan, regex=True)
    df = df.replace("\xa0", np.nan, regex=True)
    numeric_cols = [
        "DepthMidpoint", "Mean", "Sorting", "Skewness", "Kurtosis",
        "CN_Ratio", "MagDepth", "MagSus", "L_Depth", "L_star", "Fe",
    ]
    existing = [c for c in numeric_cols if c in df.columns]
    df[existing] = df[existing].apply(pd.to_numeric, errors="coerce")
    return df


def split_proxies(df):
    """
    Return a dict of tidy (depth, value) DataFrames, one per proxy.
    Depths are taken from the column that actually holds that proxy's depth.
    """
    proxies = {}
    if "Mean" in df.columns:
        proxies["Mean"] = df[["DepthMidpoint", "Mean"]].dropna().rename(
            columns={"DepthMidpoint": "depth", "Mean": "value"})
    if "CN_Ratio" in df.columns:
        proxies["CN_Ratio"] = df[["DepthMidpoint", "CN_Ratio"]].dropna().rename(
            columns={"DepthMidpoint": "depth", "CN_Ratio": "value"})
    if "MagSus" in df.columns:
        proxies["MagSus"] = df[["MagDepth", "MagSus"]].dropna().rename(
            columns={"MagDepth": "depth", "MagSus": "value"})
    if "L_star" in df.columns:
        proxies["L_star"] = df[["L_Depth", "L_star"]].dropna().rename(
            columns={"L_Depth": "depth", "L_star": "value"})
    if "Fe" in df.columns:
        proxies["Fe"] = df[["MagDepth", "Fe"]].dropna().rename(
            columns={"MagDepth": "depth", "Fe": "value"})
    return proxies

# =============================================================================
# SHARED PLOTTING HELPERS
# =============================================================================

def _shade_events(ax, events, orientation="vertical"):
    """
    Shade event deposit intervals on an axis.

    Parameters
    ----------
    ax : matplotlib Axes
    events : list of (min_depth, max_depth) tuples
    orientation : 'vertical' — depth is on the Y axis (standard stratigraphy plots)
    """
    for (d_min, d_max) in events:
        ax.axhspan(d_min, d_max, color="grey", alpha=0.25, zorder=0)


def _rescale_x_to_y_window(ax, x_series, y_series, y_lo, y_hi, pad_frac=0.05):
    """Rescale x-axis limits to only the data visible in the current y window."""
    mask = (y_series >= y_lo) & (y_series <= y_hi)
    visible_x = x_series[mask]
    if visible_x.empty:
        return
    pad = (visible_x.max() - visible_x.min()) * pad_frac
    ax.set_xlim(visible_x.min() - pad, visible_x.max() + pad)


def _event_legend_handle():
    return Patch(facecolor="grey", alpha=0.25, label="Event deposit")

# =============================================================================
# PLOT 1 — Multi-panel proxy stratigraphy
# =============================================================================

def plot_proxy_panels(proxies, title, events=(), depth_window=None,
                      proxy_order=None, save_path=None):
    """
    Plot each proxy in its own panel, sharing the Y (depth) axis.

    Parameters
    ----------
    proxies : dict returned by split_proxies()
    title : str — figure suptitle
    events : list of (min_depth, max_depth) tuples
    depth_window : (y_lo, y_hi) tuple to zoom in, or None for full core
    proxy_order : list of proxy keys controlling panel order; defaults to dict order
    save_path : file path to save figure, or None
    """
    keys = proxy_order or [k for k in proxies]
    n = len(keys)

    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 10), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, key in zip(axes, keys):
        pdata = proxies[key]
        colour = PROXY_COLOURS.get(key, "black")
        label = PROXY_CONFIG.get(key, key)

        ax.plot(pdata["value"], pdata["depth"],
                color=colour, linewidth=1, marker="o", markersize=3)
        ax.set_xlabel(label)
        ax.set_title(label, fontsize=9)

        if events:
            _shade_events(ax, events)

    axes[0].set_ylabel("Depth (mm)")
    axes[0].invert_yaxis()

    if depth_window is not None:
        y_lo, y_hi = depth_window
        axes[0].set_ylim(y_hi, y_lo)   # inverted axis: larger value at bottom
        for ax, key in zip(axes, keys):
            pdata = proxies[key]
            _rescale_x_to_y_window(ax, pdata["value"], pdata["depth"], y_lo, y_hi)

    # Add event legend if needed
    if events:
        axes[-1].legend(handles=[_event_legend_handle()],
                        loc="lower right", fontsize=8)

    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# =============================================================================
# PLOT 2 — Normalised overlay (all proxies on one panel)
# =============================================================================

def plot_normalised_overlay(proxies, title, events=(), proxy_order=None,
                            save_path=None):
    """
    Normalise all proxies to [0, 1] and overlay them on a single stratigraphy panel.
    """
    keys = proxy_order or list(proxies)
    scaler = MinMaxScaler()

    fig, ax = plt.subplots(figsize=(6, 12))

    for key in keys:
        pdata = proxies[key].copy()
        pdata["norm"] = scaler.fit_transform(pdata[["value"]])
        colour = PROXY_COLOURS.get(key, "black")
        label = PROXY_CONFIG.get(key, key)
        ax.plot(pdata["norm"], pdata["depth"],
                color=colour, linewidth=0.8, alpha=0.9, label=label)

    if events:
        _shade_events(ax, events)

    ax.invert_yaxis()
    ax.set_ylabel("Depth (mm)")
    ax.set_xlabel("Normalised Value (0–1)")
    ax.set_title(title)

    handles, labels = ax.get_legend_handles_labels()
    if events:
        handles.append(_event_legend_handle())
        labels.append("Event deposit")
    ax.legend(handles=handles, labels=labels, loc="lower right", fontsize=8)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# =============================================================================
# PLOT 3 — Dual-axis overlay (two proxies, independent x scales)
# =============================================================================

def plot_dual_axis(proxies, key_a, key_b, title, events=(), save_path=None):
    """
    Plot two proxies on twin x-axes sharing one y (depth) axis.
    Useful for directly comparing grain size and C/N ratio.
    """
    fig, ax1 = plt.subplots(figsize=(5, 12))

    pdata_a = proxies[key_a]
    pdata_b = proxies[key_b]
    colour_a = PROXY_COLOURS.get(key_a, "steelblue")
    colour_b = PROXY_COLOURS.get(key_b, "darkorange")
    label_a = PROXY_CONFIG.get(key_a, key_a)
    label_b = PROXY_CONFIG.get(key_b, key_b)

    ax1.plot(pdata_a["value"], pdata_a["depth"],
             color=colour_a, linewidth=0.8, label=label_a)
    ax1.set_xlabel(label_a, color=colour_a)
    ax1.tick_params(axis="x", labelcolor=colour_a)
    ax1.set_ylabel("Depth (mm)")
    ax1.invert_yaxis()

    ax2 = ax1.twiny()
    ax2.plot(pdata_b["value"], pdata_b["depth"],
             color=colour_b, linewidth=0.8, marker="o", markersize=2, label=label_b)
    ax2.set_xlabel(label_b, color=colour_b)
    ax2.tick_params(axis="x", labelcolor=colour_b)

    if events:
        _shade_events(ax1, events)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    extra = [_event_legend_handle()] if events else []
    ax1.legend(lines1 + lines2 + extra,
               labels1 + labels2 + (["Event deposit"] if events else []),
               loc="lower right", fontsize=8)

    plt.title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# =============================================================================
# PLOT 4 — Interpolated proxy correlation matrix
# =============================================================================

def plot_correlation_matrix(proxies, title, save_path=None):
    """
    Interpolate all proxies to a common depth grid and plot a Pearson
    correlation heatmap.
    """
    # Common depth range across all proxies
    depth_min = max(p["depth"].min() for p in proxies.values())
    depth_max = min(p["depth"].max() for p in proxies.values())
    common_depth = np.linspace(depth_min, depth_max, 500)

    aligned = {}
    for key, pdata in proxies.items():
        f = interp1d(pdata["depth"], pdata["value"],
                     bounds_error=False, fill_value=np.nan)
        label = PROXY_CONFIG.get(key, key)
        aligned[label] = f(common_depth)

    df_aligned = pd.DataFrame(aligned, index=common_depth).dropna()
    corr = df_aligned.corr(method="pearson")

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                vmin=-1, vmax=1, ax=ax, square=True)
    ax.set_title(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

    return corr

# =============================================================================
# PLOT 5 — Raw vs interpolated check
# =============================================================================

def plot_interpolation_check(proxies, title, save_path=None):
    """
    Overlay raw scatter points and interpolated lines for a quick QC check.
    """
    depth_min = max(p["depth"].min() for p in proxies.values())
    depth_max = min(p["depth"].max() for p in proxies.values())
    common_depth = np.linspace(depth_min, depth_max, 500)

    keys = list(proxies)
    n = len(keys)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 10), sharey=True)
    if n == 1:
        axes = [axes]

    for ax, key in zip(axes, keys):
        pdata = proxies[key]
        colour = PROXY_COLOURS.get(key, "black")
        label = PROXY_CONFIG.get(key, key)

        f = interp1d(pdata["depth"], pdata["value"],
                     bounds_error=False, fill_value=np.nan)
        interp_vals = f(common_depth)

        ax.scatter(pdata["value"], pdata["depth"],
                   color=colour, s=8, alpha=0.4, label="Raw")
        ax.plot(interp_vals, common_depth,
                color=colour, linewidth=1.2, alpha=0.9, label="Interpolated")
        ax.set_xlabel(label)
        ax.legend(fontsize=7)

    axes[0].set_ylabel("Depth (mm)")
    axes[0].invert_yaxis()
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# =============================================================================
# RUN — load data and generate all plots
# =============================================================================

# --- Core 13 ---
df13 = load_core13("Core13.xlsx")
p13  = split_proxies(df13)

plot_proxy_panels(
    p13, title="Core 13 — Proxy Records",
    events=EVENT_DEPOSITS_C13,
    proxy_order=["Mean", "CN_Ratio", "MagSus", "L_star"],
    save_path="Core13_panels.png",
)

plot_normalised_overlay(
    p13, title="Core 13 — All Proxies Overlaid",
    events=EVENT_DEPOSITS_C13,
    proxy_order=["Mean", "CN_Ratio", "MagSus", "L_star"],
    save_path="Core13_overlaid.png",
)

plot_dual_axis(
    p13, key_a="Mean", key_b="CN_Ratio",
    title="Core 13 — Grain Size vs C/N Ratio",
    events=EVENT_DEPOSITS_C13,
    save_path="Core13_GSvsCN.png",
)

plot_correlation_matrix(
    {k: p13[k] for k in ["Mean", "CN_Ratio", "MagSus", "L_star"] if k in p13},
    title="Core 13 — Proxy Correlation Matrix",
    save_path="Core13_correlation.png",
)

# Zoomed panel example — edit depth_window to focus on a specific interval
plot_proxy_panels(
    p13, title="Core 13 — Proxy Records (zoomed)",
    events=EVENT_DEPOSITS_C13,
    proxy_order=["Mean", "CN_Ratio", "MagSus", "L_star"],
    depth_window=(320, 420),
    save_path="Core13_panels_zoomed.png",
)

# --- Core 17 ---
df17 = load_core17("Core17.xlsx")
p17  = split_proxies(df17)

plot_proxy_panels(
    p17, title="Core 17 — Proxy Records",
    events=EVENT_DEPOSITS_C17,
    proxy_order=["Mean", "CN_Ratio", "MagSus", "L_star"],
    save_path="Core17_panels.png",
)

plot_normalised_overlay(
    p17, title="Core 17 — All Proxies Overlaid",
    events=EVENT_DEPOSITS_C17,
    proxy_order=["Mean", "CN_Ratio", "MagSus", "L_star"],
    save_path="Core17_overlaid.png",
)

plot_dual_axis(
    p17, key_a="Mean", key_b="CN_Ratio",
    title="Core 17 — Grain Size vs C/N Ratio",
    events=EVENT_DEPOSITS_C17,
    save_path="Core17_GSvsCN.png",
)

plot_correlation_matrix(
    {k: p17[k] for k in ["Mean", "CN_Ratio", "MagSus", "L_star"] if k in p17},
    title="Core 17 — Proxy Correlation Matrix",
    save_path="Core17_correlation.png",
)

plot_proxy_panels(
    p17, title="Core 17 — Proxy Records (zoomed)",
    events=EVENT_DEPOSITS_C17,
    proxy_order=["Mean", "CN_Ratio", "MagSus", "L_star"],
    depth_window=(150, 300),
    save_path="Core17_panels_zoomed.png",
)
# %%
